# ConnectTel Churn: Decision Tree and Random Forest (Complete Notebook)
Run this notebook from top to bottom: **Kernel > Restart & Run All**.
Dataset: Telco Customer Churn (BlastChar), https://www.kaggle.com/datasets/blastchar/telco-customer-churn
Place the CSV at `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`.

# Part A: Problem Understanding

**Q1. What is churn?**
Churn means a customer cancels their ConnectTel subscription. One row is one customer, for example a customer with tenure 2 months, a Month-to-month contract and MonthlyCharges 53.85 who left. The inputs are the customer's profile, account/billing and service columns. The target is Churn (No = 0, Yes = 1). Because the target has two categories, the core task is classification, not regression.

**Q2. False negative vs false positive**
A false negative means the model says a customer will stay but the customer leaves, so the retention team never contacts them and the revenue is lost. A false positive means the model flags a customer who would have stayed, so the team wastes effort or a discount on them. A false negative usually costs more, but this depends on the cost of a retention offer. To decide, I would need the average revenue per customer, the cost of a retention call or discount, and how often offers actually succeed.

**Q3. Hypotheses to test (not facts)**
1. Customers on Month-to-month contracts churn more than customers on One-year or Two-year contracts.
2. Customers with short tenure churn more than long-tenure customers.
3. Customers without TechSupport or OnlineSecurity churn more than those with these services.

**Q4. Why customerID and Churn are not inputs**
customerID is only a unique label with no real meaning; a tree could memorise it and overfit without learning anything that generalises. Churn is the answer itself. Using it as an input is target leakage: the model would look perfect but be useless, because for a real customer we do not know the outcome in advance.

**Q5. What the data can and cannot establish**
The data shows associations between customer attributes and churn at one point in time. It cannot prove that a discount will prevent churn, because association is not causation. It also cannot support prospective forecasting, because we do not know when each feature was recorded or whether there is a future outcome window. This is an educational task on one (fictional) company's data, not a production-ready forecasting system.

# Part B: Manual Split Calculations (verified with Python)
| Customer | Contract | Tenure | Churn |
|---|---|---|---|
| A | Monthly | 2 | Yes |
| B | Monthly | 4 | Yes |
| C | Monthly | 8 | No |
| D | Monthly | 12 | Yes |
| E | Annual | 3 | No |
| F | Annual | 9 | No |
| G | Annual | 18 | No |
| H | Annual | 24 | Yes |

In [ ]:
import numpy as np

teaching = {
    "A": ("Monthly", 2, "Yes"), "B": ("Monthly", 4, "Yes"),
    "C": ("Monthly", 8, "No"),  "D": ("Monthly", 12, "Yes"),
    "E": ("Annual", 3, "No"),   "F": ("Annual", 9, "No"),
    "G": ("Annual", 18, "No"),  "H": ("Annual", 24, "Yes"),
}
root_labels = [v[2] for v in teaching.values()]

def gini(lbls):
    n = len(lbls)
    if n == 0:
        return 0.0
    return 1 - sum((lbls.count(c) / n) ** 2 for c in ["Yes", "No"])

def entropy(lbls):
    n = len(lbls)
    if n == 0:
        return 0.0
    total = 0.0
    for c in ["Yes", "No"]:
        p = lbls.count(c) / n
        if p > 0:
            total -= p * np.log2(p)
    return total

def evaluate_split(name, left_ids, right_ids):
    left = [teaching[c][2] for c in left_ids]
    right = [teaching[c][2] for c in right_ids]
    n = len(left) + len(right)
    w_g = len(left) / n * gini(left) + len(right) / n * gini(right)
    w_e = len(left) / n * entropy(left) + len(right) / n * entropy(right)
    print(name)
    print("  Left :", left, "Gini", round(gini(left), 4), "Entropy", round(entropy(left), 4))
    print("  Right:", right, "Gini", round(gini(right), 4), "Entropy", round(entropy(right), 4))
    print("  Weighted Gini", round(w_g, 4), "| reduction", round(gini(root_labels) - w_g, 4))
    print("  Weighted Entropy", round(w_e, 4), "| reduction", round(entropy(root_labels) - w_e, 4))

print("Root Gini:", round(gini(root_labels), 4), "| Root Entropy:", round(entropy(root_labels), 4))
evaluate_split("Split 1: Contract Monthly vs Annual",
               [c for c, v in teaching.items() if v[0] == "Monthly"],
               [c for c, v in teaching.items() if v[0] == "Annual"])
evaluate_split("Split 2: Tenure <= 6 vs > 6",
               [c for c, v in teaching.items() if v[1] <= 6],
               [c for c, v in teaching.items() if v[1] > 6])

## Part B conclusions

| Split | Weighted Gini | Gini reduction | Weighted Entropy | Entropy reduction |
|---|---|---|---|---|
| Contract (Monthly vs Annual) | 0.375 | 0.125 | 0.8113 | 0.1887 |
| Tenure <= 6 vs > 6 | 0.4667 | 0.0333 | 0.9512 | 0.0488 |

**Decision:** Contract is the better root under both criteria (larger impurity reduction).

    Contract = Monthly ?                <- root node
     |-- Yes -> LEAF: Churn = Yes, P(Yes) = 3/4 = 0.75
     |-- No  -> Tenure <= 18 ?          <- internal (decision) node
                 |-- Yes -> LEAF (E,F,G): No,  P(Yes) = 0/3 = 0.00
                 |-- No  -> LEAF (H):     Yes, P(Yes) = 1/1 = 1.00

- **Root node** = first question; **internal node** = a later question; **branch** = the answer line; **leaf** = final prediction box.
- A split is a question about an input because the tree can only ask about the input columns.
- A leaf predicts its majority class; its probability is the class proportion. A tied leaf is predicted as the first sorted class (No) in scikit-learn; for churn I would flag it as Yes because a false negative usually costs more.
- Greedy choice looks only at the current step, so it does not guarantee the globally best tree. A numerical tree tests candidate thresholds between sorted values instead of picking a "sensible" one.

# Part C: Data Audit and Leakage-Free Setup

In [ ]:
import sys, platform, os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, sklearn

warnings.filterwarnings("ignore")
SEED = 42
NOTEBOOK_START = time.time()
DATA_PATH = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
print("Processor:", platform.processor(), "| CPU cores:", os.cpu_count(), "| GPU used: none")
for lib in [pd, np, sklearn, sns, joblib]:
    print(lib.__name__, lib.__version__)
print("Seed:", SEED)

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values (NaN):", int(df.isnull().sum().sum()))

blank_counts = df.astype(str).apply(lambda s: (s.str.strip() == "").sum())
print("\nBlank strings per column (only columns with blanks):")
print(blank_counts[blank_counts > 0])

print("\nDuplicate rows:", int(df.duplicated().sum()))
print("Repeated customerIDs:", int(df["customerID"].duplicated().sum()))
print("\nChurn counts:")
print(df["Churn"].value_counts())
print("\nChurn percentages:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

In [ ]:
tc_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
invalid_rows = df[tc_numeric.isnull()]
print("Invalid TotalCharges entries:", len(invalid_rows))
print("Raw values:", invalid_rows["TotalCharges"].unique())
display(invalid_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

In [ ]:
quality = pd.DataFrame({
    "Check": ["Rows", "Columns", "NaN cells", "Blank TotalCharges", "Duplicate rows", "Repeated customerIDs"],
    "Result": [df.shape[0], df.shape[1], int(df.isnull().sum().sum()),
               int((df["TotalCharges"].astype(str).str.strip() == "").sum()),
               int(df.duplicated().sum()), int(df["customerID"].duplicated().sum())],
})
display(quality)

counts = df["Churn"].value_counts()
plt.figure(figsize=(4, 3))
plt.bar(counts.index, counts.values, color=["steelblue", "tomato"])
for i, v in enumerate(counts.values):
    plt.text(i, v + 40, f"{v} ({v / len(df) * 100:.1f}%)", ha="center")
plt.title("Churn class distribution")
plt.ylabel("Customers")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Deterministic fixes only (nothing is learned from data here)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})  # a category, not a number

ids = df["customerID"]
y = (df["Churn"] == "Yes").astype(int)            # positive class: Churn = 1
X = df.drop(columns=["customerID", "Churn"])

X_dev, X_test, y_dev, y_test, ids_dev, ids_test = train_test_split(
    X, y, ids, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_dev, y_dev, ids_dev, test_size=0.25, stratify=y_dev, random_state=SEED)

for name, part in [("Train", y_train), ("Validation", y_val), ("Test (sealed)", y_test)]:
    print(f"{name:14s} rows={len(part):5d}  churn rate={part.mean():.4f}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score)

results_log = []   # every classification experiment is stored here

def run_experiment(name, model, params="", log=True):
    pipe = Pipeline([("prep", clone(preprocessor)), ("model", model)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fit_sec = time.time() - t0
    train_pred = pipe.predict(X_train)
    val_pred = pipe.predict(X_val)
    val_proba = pipe.predict_proba(X_val)[:, 1]
    est = pipe.named_steps["model"]
    row = {
        "Experiment": name, "Params": params, "Threshold": 0.5,
        "Train F1": f1_score(y_train, train_pred),
        "Val F1": f1_score(y_val, val_pred),
        "Val Precision": precision_score(y_val, val_pred, zero_division=0),
        "Val Recall": recall_score(y_val, val_pred),
        "Val Avg Precision": average_precision_score(y_val, val_proba),
        "Fit sec": fit_sec,
        "Depth": est.get_depth() if hasattr(est, "get_depth") else None,
        "Leaves": est.get_n_leaves() if hasattr(est, "get_n_leaves") else None,
        "Trees": getattr(est, "n_estimators", None),
    }
    if log:
        results_log.append(row)
    return pipe, row

dummy_pipe, dummy_row = run_experiment("Dummy baseline", DummyClassifier(strategy="most_frequent"),
                                       "most_frequent")
dummy_acc = accuracy_score(y_val, dummy_pipe.predict(X_val))
print("Dummy validation accuracy:", round(dummy_acc, 4))
display(pd.DataFrame([dummy_row]).round(4).T)

## Part C notes
**Invalid TotalCharges:** the 11 blank values belong to customers with tenure 0 (new customers with no first bill yet). They are converted to missing and imputed with the training median inside the pipeline, not deleted.

**Duplicates / repeated IDs:** none were found, so a normal stratified split is valid; no grouped split is needed.

**Split:** 60% train, 20% validation, 20% sealed test, stratified, random_state=42. The test set is not used for any choice.

**No leakage:** imputation and one-hot encoding are inside the Pipeline, fitted only on training rows (and only on training folds in cross-validation).

**Why trees need no standardisation:** a split asks "feature <= threshold?", which depends on the order of values, not their scale. Arbitrary numeric category codes would invent an order that does not exist, so one-hot encoding is used.

**Why accuracy misleads:** the dummy model predicts "No" for everybody and still gets about 73% accuracy, yet finds no churners (recall 0).

# Part D: Build, Explain and Prune a Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# D1: unrestricted tree
tree_full, row = run_experiment("Tree unrestricted", DecisionTreeClassifier(random_state=SEED), "max_depth=None")
display(pd.DataFrame([row]).round(4).T)

In [ ]:
# D2: depth curve
depths = [1, 2, 3, 5, 10, None]
depth_rows = []
for d in depths:
    _, r = run_experiment(f"Tree depth={d}", DecisionTreeClassifier(max_depth=d, random_state=SEED), f"max_depth={d}")
    depth_rows.append(r)
depth_df = pd.DataFrame(depth_rows)
display(depth_df[["Params", "Train F1", "Val F1", "Depth", "Leaves", "Fit sec"]].round(4))

plt.figure(figsize=(6, 4))
plt.plot([str(d) for d in depths], depth_df["Train F1"], marker="o", label="Train F1")
plt.plot([str(d) for d in depths], depth_df["Val F1"], marker="o", label="Validation F1")
plt.xlabel("max_depth"); plt.ylabel("Churn F1"); plt.title("Decision Tree: F1 vs depth")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# D3: min_samples_leaf and min_samples_split at max_depth=5
rows = []
for msl in [1, 5, 20]:
    _, r = run_experiment(f"Tree d5 msl={msl}", DecisionTreeClassifier(max_depth=5, min_samples_leaf=msl, random_state=SEED),
                          f"max_depth=5, min_samples_leaf={msl}")
    rows.append(r)
for mss in [2, 20, 50]:
    _, r = run_experiment(f"Tree d5 mss={mss}", DecisionTreeClassifier(max_depth=5, min_samples_split=mss, random_state=SEED),
                          f"max_depth=5, min_samples_split={mss}")
    rows.append(r)
display(pd.DataFrame(rows)[["Params", "Train F1", "Val F1", "Val Precision", "Val Recall", "Leaves"]].round(4))

In [ ]:
# D4: gini vs entropy
rows = []
for crit in ["gini", "entropy"]:
    _, r = run_experiment(f"Tree d5 {crit}", DecisionTreeClassifier(max_depth=5, criterion=crit, random_state=SEED),
                          f"max_depth=5, criterion={crit}")
    rows.append(r)
display(pd.DataFrame(rows)[["Params", "Train F1", "Val F1", "Val Precision", "Val Recall", "Leaves"]].round(4))

In [ ]:
# D5: cost-complexity post-pruning (alphas come from the TRAINING partition only)
prep_train = clone(preprocessor).fit(X_train)
Xt_train = prep_train.transform(X_train)
path = DecisionTreeClassifier(random_state=SEED).cost_complexity_pruning_path(Xt_train, y_train)
all_alphas = np.unique(path.ccp_alphas)
usable = all_alphas[:-1] if len(all_alphas) > 1 else all_alphas      # last alpha prunes to a single node
pick = np.unique(np.round(np.linspace(0, len(usable) - 1, 8)).astype(int))
chosen_alphas = usable[pick]
print("Chosen ccp_alpha values (max 8, includes 0):", np.round(chosen_alphas, 6))

pruned_pipes, pruned_rows = {}, []
for a in chosen_alphas:
    p, r = run_experiment(f"Tree pruned alpha={a:.5f}", DecisionTreeClassifier(ccp_alpha=a, random_state=SEED),
                          f"ccp_alpha={a:.5f}")
    pruned_pipes[a] = p
    pruned_rows.append(r)
pruned_df = pd.DataFrame(pruned_rows)
display(pruned_df[["Params", "Train F1", "Val F1", "Depth", "Leaves"]].round(4))

best_alpha = chosen_alphas[int(pruned_df["Val F1"].idxmax())]
best_pruned_pipe = pruned_pipes[best_alpha]
print("Best ccp_alpha by validation F1:", round(best_alpha, 6))

fig, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(chosen_alphas, pruned_df["Val F1"], marker="o", color="tab:blue")
ax1.set_xlabel("ccp_alpha"); ax1.set_ylabel("Validation F1", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(chosen_alphas, pruned_df["Leaves"], marker="s", color="tab:red")
ax2.set_ylabel("Leaves", color="tab:red")
plt.title("Cost-complexity pruning"); plt.show()

In [ ]:
# D6: readable shallow tree (depth 3)
tree3, _ = run_experiment("Tree depth=3 (plot)", DecisionTreeClassifier(max_depth=3, random_state=SEED), "max_depth=3", log=False)
feat_names = tree3.named_steps["prep"].get_feature_names_out()
plt.figure(figsize=(22, 9))
plot_tree(tree3.named_steps["model"], feature_names=feat_names, class_names=["Stay", "Churn"],
          filled=True, rounded=True, fontsize=8)
plt.title("Shallow tree (max_depth=3)"); plt.show()

In [ ]:
# D7: trace two validation customers (one churner, one stayer)
def explain_customer(pipe, row_df, true_label):
    prep, est = pipe.named_steps["prep"], pipe.named_steps["model"]
    Xt = prep.transform(row_df)
    names = prep.get_feature_names_out()
    dense = Xt.toarray() if hasattr(Xt, "toarray") else np.asarray(Xt)
    nodes = est.decision_path(Xt).indices
    leaf = est.apply(Xt)[0]
    print(f"Actual churn = {true_label}")
    for node in nodes:
        if node == leaf:
            break
        f, thr = est.tree_.feature[node], est.tree_.threshold[node]
        val = dense[0, f]
        print(f"  node {node}: {names[f]} = {val:.2f} -> {'<=' if val <= thr else '>'} {thr:.2f}")
    print("  leaf", leaf, "| samples in leaf:", est.tree_.n_node_samples[leaf],
          "| leaf class values:", np.round(est.tree_.value[leaf][0], 3))
    print("  predict_proba [stay, churn]:", np.round(pipe.predict_proba(row_df)[0], 3))
    print("  (probability = class proportion inside the leaf; no sample weights used here)\n")

pos_churn = int(np.where(y_val.values == 1)[0][0])
pos_stay = int(np.where(y_val.values == 0)[0][0])
for pos in [pos_churn, pos_stay]:
    print("Validation customer", ids_val.iloc[pos])
    explain_customer(tree3, X_val.iloc[[pos]], int(y_val.iloc[pos]))

In [ ]:
# D8: stability on three different splits made inside the development data
def three_splits(make_model, label):
    out = []
    for s in [1, 2, 3]:
        Xtr, Xva, ytr, yva = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=s)
        pipe = Pipeline([("prep", clone(preprocessor)), ("model", make_model())])
        t0 = time.time(); pipe.fit(Xtr, ytr); sec = time.time() - t0
        est = pipe.named_steps["model"]
        root = ""
        if hasattr(est, "tree_"):
            root = pipe.named_steps["prep"].get_feature_names_out()[est.tree_.feature[0]]
        out.append({"Model": label, "Split seed": s, "Val F1": f1_score(yva, pipe.predict(Xva)),
                    "Root feature": root, "Fit sec": sec})
    return pd.DataFrame(out)

stab_tree = three_splits(lambda: DecisionTreeClassifier(max_depth=5, random_state=SEED), "Tree depth5")
display(stab_tree.round(4))
print("Tree F1 std over 3 splits:", round(stab_tree["Val F1"].std(), 4))

## Part D notes (fill the blanks from YOUR outputs above)
**D1:** the unrestricted tree has train F1 = ___, validation F1 = ___. The gap shows overfitting because the tree memorised training customers.

**D2:** the best validation F1 is at max_depth = ___. Depth 1 underfits (too simple to separate churners); depth None overfits (train F1 near 1, validation much lower).

**D3:** min_samples_leaf stops leaves with fewer than that many customers; min_samples_split stops splitting nodes smaller than that. Bigger values gave fewer leaves and validation F1 changed from ___ to ___. max_leaf_nodes caps the number of leaves; min_impurity_decrease blocks splits that reduce impurity too little (also pre-pruning).

**D4:** gini F1 = ___, entropy F1 = ___. They are close because both measure impurity similarly. log_loss is the same idea as entropy.

**D5:** pre-pruning restricts growth while the tree is built; post-pruning (ccp_alpha) grows a full tree and then removes weak branches. Best alpha = ___ with ___ leaves and validation F1 = ___.

**D6/D7:** the root condition is the first feature/threshold in the plot; `gini` is the impurity of the node; `samples` is the customers reaching it; `value` is class proportions (or counts, depending on version); the prediction is the majority class. Each customer follows the printed conditions to a leaf, and predict_proba equals the churn proportion in that leaf.

**D8:** root feature and F1 change between splits (std = ___), showing a single tree is sensitive to the training sample.

# Part E: Build and Inspect a Random Forest

In [ ]:
# E1: three bootstrap samples on the 8-row teaching table
rng = np.random.RandomState(SEED)
letters = list(teaching.keys())
boot_rows = []
for i in range(1, 4):
    idx = rng.randint(0, len(letters), len(letters))
    sample = [letters[j] for j in idx]
    oob = [c for c in letters if c not in sample]
    boot_rows.append({"Sample": i, "Bootstrap IDs": ", ".join(sample),
                      "Unique IDs": len(set(sample)), "Out-of-bag IDs": ", ".join(oob)})
display(pd.DataFrame(boot_rows))
print("Large-n unique fraction: 1 - (1 - 1/n)^n ->", round(1 - (1 - 1 / 100000) ** 100000, 4), "(about 63.2%)")

In [ ]:
# E2: Random Forest with OOB score
from sklearn.ensemble import RandomForestClassifier

rf_pipe, rf_row = run_experiment(
    "RF 100 trees (oob)",
    RandomForestClassifier(n_estimators=100, bootstrap=True, oob_score=True, random_state=SEED, n_jobs=2),
    "n_estimators=100, bootstrap=True, oob_score=True")
display(pd.DataFrame([rf_row]).round(4).T)

In [ ]:
# E3: aggregation check on one validation customer
rf_prep, rf_model = rf_pipe.named_steps["prep"], rf_pipe.named_steps["model"]
one = X_val.iloc[[0]]
Xt_one = rf_prep.transform(one)
tree_probs = np.array([t.predict_proba(Xt_one)[0] for t in rf_model.estimators_])
print("Per-tree churn probabilities (first 10 trees):", np.round(tree_probs[:10, 1], 3))
avg_proba = tree_probs.mean(axis=0)
print("Average of per-tree probabilities:", np.round(avg_proba, 4))
print("Forest predict_proba            :", np.round(rf_model.predict_proba(Xt_one)[0], 4))
print("Match:", np.allclose(avg_proba, rf_model.predict_proba(Xt_one)[0]))
hard_votes = tree_probs.argmax(axis=1)
print("Hard-vote churn share:", round(hard_votes.mean(), 3), "vs averaged churn probability:", round(avg_proba[1], 3))

In [ ]:
# E4: number of trees
rows = []
for n in [10, 50, 100, 200]:
    _, r = run_experiment(f"RF n_estimators={n}", RandomForestClassifier(n_estimators=n, random_state=SEED, n_jobs=2),
                          f"n_estimators={n}")
    rows.append(r)
tree_count_df = pd.DataFrame(rows)
display(tree_count_df[["Params", "Train F1", "Val F1", "Fit sec"]].round(4))

fig, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(tree_count_df["Trees"], tree_count_df["Val F1"], marker="o", color="tab:blue")
ax1.set_xlabel("n_estimators"); ax1.set_ylabel("Validation F1", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(tree_count_df["Trees"], tree_count_df["Fit sec"], marker="s", color="tab:red")
ax2.set_ylabel("Fit seconds", color="tab:red")
plt.title("Forest: tree count vs F1 and time"); plt.show()

In [ ]:
# E5: max_features
rows = []
for mf in ["sqrt", 0.5, None]:
    _, r = run_experiment(f"RF max_features={mf}", RandomForestClassifier(n_estimators=100, max_features=mf, random_state=SEED, n_jobs=2),
                          f"max_features={mf}")
    rows.append(r)
display(pd.DataFrame(rows)[["Params", "Train F1", "Val F1", "Val Precision", "Val Recall", "Fit sec"]].round(4))

In [ ]:
# E6: bootstrap on/off and max_samples
rows = []
_, r = run_experiment("RF bootstrap=True", RandomForestClassifier(n_estimators=100, bootstrap=True, random_state=SEED, n_jobs=2), "bootstrap=True"); rows.append(r)
_, r = run_experiment("RF bootstrap=False", RandomForestClassifier(n_estimators=100, bootstrap=False, oob_score=False, random_state=SEED, n_jobs=2), "bootstrap=False"); rows.append(r)
_, r = run_experiment("RF max_samples=0.7", RandomForestClassifier(n_estimators=100, bootstrap=True, max_samples=0.7, random_state=SEED, n_jobs=2), "bootstrap=True, max_samples=0.7"); rows.append(r)
_, r = run_experiment("RF max_samples=None", RandomForestClassifier(n_estimators=100, bootstrap=True, max_samples=None, random_state=SEED, n_jobs=2), "bootstrap=True, max_samples=None"); rows.append(r)
display(pd.DataFrame(rows)[["Params", "Train F1", "Val F1", "Fit sec"]].round(4))

In [ ]:
# E7/E8: OOB analysis
oob_acc = rf_model.oob_score_                      # default OOB metric is ACCURACY, not F1
oob_prob = rf_model.oob_decision_function_[:, 1]
valid = ~np.isnan(oob_prob)
oob_pred = (oob_prob[valid] >= 0.5).astype(int)
oob_f1 = f1_score(y_train.values[valid], oob_pred)
print("OOB accuracy (oob_score_):", round(oob_acc, 4))
print("OOB churn F1 (from oob_decision_function_):", round(oob_f1, 4))
print("Rows without usable OOB prediction:", int((~valid).sum()), "of", len(valid))

oob_table = pd.DataFrame({
    "Measure": ["Train F1", "OOB F1", "Validation F1", "OOB accuracy (different metric)"],
    "Value": [rf_row["Train F1"], oob_f1, rf_row["Val F1"], oob_acc]})
display(oob_table.round(4))

In [ ]:
# E9: forest stability on the same three development-only splits
stab_rf = three_splits(lambda: RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=2), "RF 100 trees")
display(stab_rf.round(4))
print("Forest F1 std:", round(stab_rf["Val F1"].std(), 4), "| Tree F1 std:", round(stab_tree["Val F1"].std(), 4))

## Part E notes (fill the blanks from YOUR outputs above)
**E1:** each bootstrap sample has 8 draws with replacement, so some IDs repeat and some are left out (out-of-bag). For large n, a full-size bootstrap contains about 1 - 1/e = 63.2% unique rows.

**E2/E3:** every tree gets its own bootstrap sample and a random subset of candidate features at each split (not one fixed subset per tree). The forest probability is the average of per-tree probabilities (match = True above), which can differ from a hard majority vote.

**E4:** validation F1 changed from ___ (10 trees) to ___ (200 trees) while fit time grew. Returns diminish because new trees add little once the average is stable; more trees do not guarantee better validation results.

**E5:** max_features = ___ gave the best validation F1. Fewer features per split make trees weaker but less correlated; all features make strong but similar trees. Bootstrap alone resamples rows only, so trees still pick the same strong features and stay correlated.

**E6:** bootstrap=True vs False gave ___ vs ___; max_samples 0.7 vs None gave ___ vs ___.

**E7/E8:** oob_score_ is ACCURACY (___); OOB F1 = ___. OOB rows were not seen by the tree that predicts them, so it is an internal diagnostic, not a replacement for the sealed test set.

**E9:** forest F1 varies less across splits than the single tree (std ___ vs ___) but takes longer to fit. Averaging lowers variance; a forest can still overfit.

# Part F: Tune, Evaluate and Explain

In [ ]:
# F1: class_weight comparison for both families
rows = []
for cw in [None, "balanced"]:
    _, r = run_experiment(f"Tree d5 cw={cw}", DecisionTreeClassifier(max_depth=5, class_weight=cw, random_state=SEED), f"max_depth=5, class_weight={cw}"); rows.append(r)
    _, r = run_experiment(f"RF d10 cw={cw}", RandomForestClassifier(n_estimators=100, max_depth=10, class_weight=cw, random_state=SEED, n_jobs=2), f"n=100, max_depth=10, class_weight={cw}"); rows.append(r)
display(pd.DataFrame(rows)[["Experiment", "Params", "Val F1", "Val Precision", "Val Recall", "Val Avg Precision"]].round(4))

In [ ]:
# F2-F4: RandomizedSearchCV on the TRAINING partition only (pipeline inside the search)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

tree_search = RandomizedSearchCV(
    Pipeline([("prep", clone(preprocessor)), ("model", DecisionTreeClassifier(random_state=SEED))]),
    param_distributions={
        "model__max_depth": [3, 5, 10, None], "model__min_samples_leaf": [1, 5, 20],
        "model__min_samples_split": [2, 10, 30], "model__criterion": ["gini", "entropy"],
        "model__class_weight": [None, "balanced"]},
    n_iter=8, scoring="f1", cv=cv, random_state=SEED, n_jobs=2)
t0 = time.time(); tree_search.fit(X_train, y_train); tree_search_sec = time.time() - t0

forest_search = RandomizedSearchCV(
    Pipeline([("prep", clone(preprocessor)),
              ("model", RandomForestClassifier(bootstrap=True, random_state=SEED, n_jobs=1))]),
    param_distributions={
        "model__n_estimators": [50, 100, 200], "model__max_depth": [5, 10, None],
        "model__min_samples_leaf": [1, 5, 10], "model__max_features": ["sqrt", 0.5],
        "model__class_weight": [None, "balanced"]},
    n_iter=8, scoring="f1", cv=cv, random_state=SEED, n_jobs=2)
t0 = time.time(); forest_search.fit(X_train, y_train); forest_search_sec = time.time() - t0

print(f"Tree search: best CV F1 = {tree_search.best_score_:.4f} in {tree_search_sec:.1f}s")
print("  ", tree_search.best_params_)
print(f"Forest search: best CV F1 = {forest_search.best_score_:.4f} in {forest_search_sec:.1f}s")
print("  ", forest_search.best_params_)

In [ ]:
# F5: compare candidates on validation (pruning result is a separate candidate)
candidates = {
    "Tuned tree": tree_search.best_estimator_,
    "Pruned tree (best ccp_alpha)": best_pruned_pipe,
    "Tuned forest": forest_search.best_estimator_,
}
cand_rows = []
for name, pipe in candidates.items():
    vp = pipe.predict(X_val)
    vpr = pipe.predict_proba(X_val)[:, 1]
    cand_rows.append({"Candidate": name, "Val F1": f1_score(y_val, vp),
                      "Val Precision": precision_score(y_val, vp, zero_division=0),
                      "Val Recall": recall_score(y_val, vp),
                      "Val Avg Precision": average_precision_score(y_val, vpr)})
cand_df = pd.DataFrame(cand_rows)
display(cand_df.round(4))

selected_name = cand_df.loc[cand_df["Val F1"].idxmax(), "Candidate"]
selected_pipe = candidates[selected_name]
print("Selected by validation F1:", selected_name)

In [ ]:
# F5: threshold comparison on validation probabilities
val_proba_sel = selected_pipe.predict_proba(X_val)[:, 1]
thr_rows = []
for thr in [0.3, 0.5, 0.7]:
    p = (val_proba_sel >= thr).astype(int)
    thr_rows.append({"Threshold": thr, "Val F1": f1_score(y_val, p),
                     "Val Precision": precision_score(y_val, p, zero_division=0),
                     "Val Recall": recall_score(y_val, p), "Customers flagged": int(p.sum())})
thr_df = pd.DataFrame(thr_rows)
display(thr_df.round(4))
chosen_threshold = float(thr_df.loc[thr_df["Val F1"].idxmax(), "Threshold"])
print("Chosen threshold (best validation F1):", chosen_threshold)

In [ ]:
# F7: impurity importance vs permutation importance (selected candidate, validation data, before refit)
from sklearn.inspection import permutation_importance

sel_prep, sel_model = selected_pipe.named_steps["prep"], selected_pipe.named_steps["model"]
enc_names = sel_prep.get_feature_names_out()

def original_name(enc):
    if enc.startswith("num__"):
        return enc[len("num__"):]
    body = enc[len("cat__"):]
    matches = [c for c in categorical_cols if body.startswith(c + "_")]
    return max(matches, key=len) if matches else body

imp = pd.Series(sel_model.feature_importances_, index=[original_name(n) for n in enc_names])
impurity_imp = imp.groupby(level=0).sum().sort_values(ascending=False)

perm = permutation_importance(selected_pipe, X_val, y_val, scoring="f1", n_repeats=5,
                              random_state=SEED, n_jobs=1)
perm_imp = pd.Series(perm.importances_mean, index=X_val.columns).sort_values(ascending=False)

imp_df = pd.DataFrame({"Impurity importance": impurity_imp, "Permutation importance (F1 drop)": perm_imp})
imp_df = imp_df.sort_values("Permutation importance (F1 drop)", ascending=False)
display(imp_df.head(10).round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
impurity_imp.head(8)[::-1].plot.barh(ax=axes[0], color="steelblue"); axes[0].set_title("Impurity importance")
perm_imp.head(8)[::-1].plot.barh(ax=axes[1], color="darkorange"); axes[1].set_title("Permutation importance (F1)")
plt.tight_layout(); plt.show()

In [ ]:
# F6: freeze, refit on train+validation, evaluate ONCE on the sealed test set
from sklearn.metrics import (confusion_matrix, roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay)

final_pipe = clone(selected_pipe)
t0 = time.time(); final_pipe.fit(X_dev, y_dev); final_fit_sec = time.time() - t0

test_proba = final_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= chosen_threshold).astype(int)

final_table = pd.DataFrame({"Metric": ["Model", "Threshold", "Accuracy", "Churn precision", "Churn recall",
                                       "Churn F1", "ROC-AUC", "Average precision"],
                            "Test value": [selected_name, chosen_threshold, accuracy_score(y_test, test_pred),
                                           precision_score(y_test, test_pred), recall_score(y_test, test_pred),
                                           f1_score(y_test, test_pred), roc_auc_score(y_test, test_proba),
                                           average_precision_score(y_test, test_proba)]})
display(final_table)

cm = confusion_matrix(y_test, test_pred)
print("Confusion matrix [rows = actual Stay/Churn, cols = predicted Stay/Churn]:\n", cm)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Stay", "Churn"], yticklabels=["Stay", "Churn"])
axes[0].set_title("Test confusion matrix"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
RocCurveDisplay.from_predictions(y_test, test_proba, ax=axes[1]); axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, test_proba, ax=axes[2]); axes[2].set_title("Precision-recall curve")
plt.tight_layout(); plt.show()

In [ ]:
# F8: error analysis on the final test set (for future work, not for more tuning)
err = X_test.copy()
err["customerID"] = ids_test.values
err["actual"] = y_test.values
err["churn_probability"] = test_proba
err["predicted"] = test_pred
cols = ["customerID", "actual", "predicted", "churn_probability", "tenure", "Contract",
        "MonthlyCharges", "InternetService", "PaymentMethod", "TechSupport"]
print("False positives (predicted churn, actually stayed):")
display(err[(err.predicted == 1) & (err.actual == 0)][cols].head(3).round(3))
print("False negatives (predicted stay, actually churned):")
display(err[(err.predicted == 0) & (err.actual == 1)][cols].head(3).round(3))

## Part F notes (fill the blanks from YOUR outputs above)
**F1:** class_weight="balanced" usually raises recall and lowers precision; here the validation F1 changed from ___ to ___. Balancing does not always help.

**F2-F4:** each search used 8 candidates, 3-fold stratified CV, scoring F1, on training data only, with preprocessing inside the pipeline. The pruning result was added as a separate validation candidate. Hardware and runtime are printed above.

**F5:** selected model = ___. Threshold 0.3 flags more customers (higher recall, lower precision); 0.7 flags fewer (higher precision, lower recall). Chosen threshold = ___ because it had the best validation F1. If a retention offer is cheap compared with a lost customer, a lower threshold is defensible.

**F6:** the frozen model was refit on train+validation and evaluated once on the test set (table above).

**F7:** impurity importance is computed from training splits and favours features with many possible split points; permutation importance measures the F1 drop on validation data. Correlated features (for example tenure and TotalCharges) share importance, so neither number proves cause.

**F8:** false positives look like ___ (customers who resemble churners but stayed); false negatives look like ___ (churners who look loyal). Importance is association, not causation.

# Part G: Regression Extension (MonthlyCharges)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

reg_y = df["MonthlyCharges"]
reg_X = df.drop(columns=["customerID", "Churn", "MonthlyCharges", "TotalCharges"])
Xr_train, Xr_val, Xr_test = reg_X.loc[X_train.index], reg_X.loc[X_val.index], reg_X.loc[X_test.index]
yr_train, yr_val, yr_test = reg_y.loc[X_train.index], reg_y.loc[X_val.index], reg_y.loc[X_test.index]

reg_num = ["tenure"]
reg_cat = [c for c in reg_X.columns if c not in reg_num]
reg_prep = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), reg_num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), reg_cat),
])

reg_models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Decision Tree (depth 5)": DecisionTreeRegressor(max_depth=5, random_state=SEED),
    "Random Forest (100, depth 10)": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=2),
}
reg_pipes, reg_rows = {}, []
for name, model in reg_models.items():
    pipe = Pipeline([("prep", clone(reg_prep)), ("model", model)])
    pipe.fit(Xr_train, yr_train)
    reg_pipes[name] = pipe
    vp = pipe.predict(Xr_val)
    reg_rows.append({"Model": name,
                     "Train MAE": mean_absolute_error(yr_train, pipe.predict(Xr_train)),
                     "Val MAE": mean_absolute_error(yr_val, vp),
                     "Val RMSE": np.sqrt(mean_squared_error(yr_val, vp)),
                     "Val R2": r2_score(yr_val, vp)})
reg_df = pd.DataFrame(reg_rows)
display(reg_df.round(4))

best_reg = reg_df.loc[reg_df["Val RMSE"].idxmin(), "Model"]
rp = reg_pipes[best_reg].predict(Xr_test)
print("Frozen regression choice (lowest validation RMSE):", best_reg)
print("One final TEST evaluation -> MAE:", round(mean_absolute_error(yr_test, rp), 4),
      "| RMSE:", round(float(np.sqrt(mean_squared_error(yr_test, rp))), 4),
      "| R2:", round(r2_score(yr_test, rp), 4))

In [ ]:
leaf = np.array([20, 30, 40])
print("Leaf prediction (mean):", leaf.mean())
print("Within-leaf MSE:", round(((leaf - leaf.mean()) ** 2).mean(), 4))

## Part G notes
**Predictors:** MonthlyCharges (target), TotalCharges, Churn and customerID are removed. TotalCharges is roughly MonthlyCharges x tenure, so it would leak the target; Churn would change the intended feature availability.

**Metrics:** a confusion matrix needs classes; regression predicts numbers, so MAE, RMSE and R-squared are used.

**Leaf maths:** values [20, 30, 40] give mean 30 and MSE = (100 + 0 + 100) / 3 = 66.67. A regression tree picks the split that lowers the weighted squared error the most; a regression forest averages the trees' predictions.

**Extrapolation:** a mean-leaf tree can only output averages of training targets it has seen, so it cannot predict beyond the observed range. Classification uses impurity of class proportions; regression uses squared error. absolute_error uses median leaves (robust to outliers); Poisson suits non-negative counts.

# Part H: Reproducible Handover

In [ ]:
joblib.dump(final_pipe, "churn_pipeline.joblib")
meta = {"threshold": chosen_threshold, "model": selected_name, "seed": SEED,
        "sklearn_version": sklearn.__version__, "positive_class": "Churn = Yes",
        "required_columns": list(X.columns)}
with open("churn_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

loaded_pipe = joblib.load("churn_pipeline.joblib")      # only load model files from trusted sources
with open("churn_metadata.json") as f:
    loaded_meta = json.load(f)

five = X_test.iloc[:5]
orig_p = final_pipe.predict_proba(five)[:, 1]
load_p = loaded_pipe.predict_proba(five)[:, 1]
print("Identical probabilities:", np.allclose(orig_p, load_p))
print("Identical decisions:", np.array_equal(orig_p >= chosen_threshold, load_p >= loaded_meta["threshold"]))

In [ ]:
def predict_customers(dataframe):
    required = loaded_meta["required_columns"]
    missing = [c for c in required if c not in dataframe.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    data = dataframe[required].copy()                       # row order preserved
    if pd.api.types.is_numeric_dtype(data["SeniorCitizen"]):
        data["SeniorCitizen"] = data["SeniorCitizen"].map({0: "No", 1: "Yes"})
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    proba = loaded_pipe.predict_proba(data)[:, 1]
    out = pd.DataFrame({"churn_probability": proba,
                        "predicted_Churn": np.where(proba >= loaded_meta["threshold"], "Yes", "No")},
                       index=dataframe.index)
    if "customerID" in dataframe.columns:
        out.insert(0, "customerID", dataframe["customerID"].values)
    return out

display(predict_customers(X_test.iloc[:3]))

new_customer = X_test.iloc[[0]].copy()
new_customer["PaymentMethod"] = "Crypto wallet"            # category never seen in training
print("Unseen PaymentMethod handled by the encoder (ignored, no crash):")
display(predict_customers(new_customer))

try:
    predict_customers(X_test.drop(columns=["Contract"]).iloc[:2])
except ValueError as e:
    print("Validation works:", e)

In [ ]:
export = pd.DataFrame({"customerID": ids_test.values,
                       "actual_Churn": np.where(y_test.values == 1, "Yes", "No"),
                       "churn_probability": test_proba,
                       "predicted_Churn": np.where(test_pred == 1, "Yes", "No")})
export.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv:", export.shape)
display(export.head())

In [ ]:
# Results table (all validation experiments) and total runtime
results_df = pd.DataFrame(results_log)
display(results_df.round(4))
results_df.to_csv("results_table.csv", index=False)
print("Total notebook runtime: %.1f seconds" % (time.time() - NOTEBOOK_START))

## Recommendation (edit with YOUR final numbers, 150-250 words)
Replace the blanks using the Part F tables.

I compared a Decision Tree and a Random Forest on the same 60/20/20 split with the same leakage-safe pipeline. The selected model was ___ with validation churn F1 = ___; on the sealed test set it reached F1 = ___, precision = ___, recall = ___, ROC-AUC = ___. The single tree is easy to read and explain to the retention team and trains quickly. The forest is usually more stable across splits but slower and harder to explain.

A false negative (a missed churner) usually costs more than a false positive (an unnecessary offer), so I chose a threshold of ___ using validation F1; a lower threshold would catch more churners at the price of more wasted offers. The retention team should prioritise customers with the highest predicted churn probability, in practice those with short tenure and Month-to-month contracts, as the importance results suggest.

Limitations: the data is a single snapshot, so it shows association, not proof that a discount prevents churn; feature timing and a future outcome window are unknown; subgroup performance and calibration were not checked; this is an educational result, not a production-ready system.